# 🎥 中文人像口播视频生成器（带 UI）

使用界面上传照片、输入中文文案，生成带配音与嘴型同步的人像口播视频。

## 🔧 技术组件
- 中文语音合成：Edge-TTS 或 Bark
- 嘴型同步：SadTalker
- 图形界面：Gradio


In [ ]:
# ✅ 安装必要依赖
!pip install --upgrade pip
!pip install edge-tts gradio imageio[ffmpeg] moviepy

# ✅ 克隆并安装 SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -r requirements.txt
%cd ..

# ✅ 安装 Bark
!pip install git+https://github.com/suno-ai/bark.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 130.3 MB/s eta 0:00:00
  DEPRECATION: Building 'srt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'srt'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=4b8b2207496f8e91e70d2a491e59b5572e615f3e344b828e5f5

In [ ]:
# ⚙️ 主要功能函数定义
import os
import shutil
import edge_tts
import asyncio
from bark import SAMPLE_RATE, generate_audio, preload_models
from scipy.io.wavfile import write as write_wav
import gradio as gr

def generate_video(image, text, use_bark):
    image_path = "input_face.png"
    audio_path = "output.wav"
    result_video = "results/results.mp4"

    # 保存图像
    image.save(image_path)

    # 生成语音
    if use_bark:
        preload_models()
        audio_array = generate_audio(text)
        write_wav(audio_path, SAMPLE_RATE, audio_array)
    else:
        async def edge_speak():
            communicate = edge_tts.Communicate(text, voice="zh-CN-XiaoxiaoNeural")
            await communicate.save(audio_path)
        asyncio.run(edge_speak())

    # 准备 SadTalker 输入
    os.makedirs("SadTalker/inputs", exist_ok=True)
    shutil.copy(image_path, "SadTalker/inputs/face.png")

    # 运行 SadTalker
    os.makedirs("results", exist_ok=True)
    os.system("cd SadTalker && python -m sadtalker --driven_audio ../output.wav --source_image inputs/face.png --result_dir ../results --still --preprocess full --enhancer gfpgan")

    # 检查输出
    if os.path.exists(result_video):
        return result_video
    else:
        return "生成失败，请检查输入或依赖。"


In [ ]:
# 🚀 启动 Gradio 界面
gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Image(type="pil", label="上传人像照片"),
        gr.Textbox(label="中文口播文案"),
        gr.Checkbox(label="使用 Bark（更自然但慢）", value=False)
    ],
    outputs=gr.Video(label="生成结果"),
    title="中文口播视频生成器",
    description="上传一张照片并输入中文口播内容，即可生成配音和嘴型同步的视频。"
).launch(share=True)
